# Tutorial: Normal Ordering Sparse Operators

---

This notebook demonstrates determinant-reference normal ordering for `SparseOperator` objects. The normal-ordering vacuum is a `Determinant`, so each alpha/beta spin orbital is either occupied or unoccupied in the reference.

In [ ]:
from forte2.lib.det import Determinant
from forte2.lib.sparse_ops import SparseState, normal_order, sparse_operator

## Choose a determinant vacuum

Here orbital 0 is doubly occupied in the vacuum. Relative to this reference, physical annihilation operators in orbital 0 create holes, while physical creation operators in orbital 0 annihilate holes.

In [ ]:
vacuum = Determinant("2")
print(vacuum.str(4))

## Build an ordinary sparse operator

The operator below mixes occupied and virtual number operators, a single excitation, and a pair excitation. It is an ordinary `SparseOperator` in the usual physical creation/annihilation representation.

In [ ]:
op = sparse_operator(
    [
        ("[0a+ 0a-]", 1.0),
        ("[1a+ 1a-]", 2.0),
        ("[1a+ 0a-]", 0.3),
        ("[1a+ 1b+ 0b- 0a-]", 0.7),
    ]
)

op.str()

## Normal order with respect to the vacuum

`normal_order(op, vacuum)` returns a `NormalOrderedSparseOperator`. Terms are printed with braces to emphasize that they are normal ordered relative to the stored reference.

In [ ]:
nop = normal_order(op, vacuum)
nop.str()

The occupied number operator `a_0^+ a_0` becomes a scalar contribution plus a hole term. Internally, `NormalOrderedString` mirrors `SQOperatorString`: it stores compact normal creators and normal annihilators as `Determinant` bitsets.

In [ ]:
for term, coeff in nop:
    print(f"{coeff:>12}  {term.str(vacuum):24s}  cre={term.cre().str(4)}  ann={term.ann().str(4)}")

## Truncate by many-body rank

The many-body rank is defined from the number of normal operators: scalar terms have rank 0, terms with one or two normal operators have rank 1, and terms with three or four normal operators have rank 2. Use `truncate(max_rank)` to discard higher-rank normal-ordered terms.

In [ ]:
for term, coeff in nop:
    print(f"rank {term.many_body_rank()}: {coeff:>12}  {term.str(vacuum)}")

one_body_nop = nop.truncate(max_rank=1)
one_body_nop.str()

You can also request the truncation directly while normal ordering.

In [ ]:
direct_one_body_nop = normal_order(op, vacuum, max_rank=1)
direct_one_body_nop == one_body_nop

## Convert back to `SparseOperator`

A normal-ordered operator can be expanded back to an ordinary `SparseOperator`. For determinant vacua this round trip is exact up to the coefficient screening threshold.

In [ ]:
expanded = nop.to_sparse_operator()

print(expanded == op)
expanded.str()

## Apply a normal-ordered operator to a sparse state

`NormalOrderedSparseOperator.apply_to_state(state)` applies the physical operators represented by each normal-ordered term. This should match applying the expanded ordinary `SparseOperator`.

In [ ]:
state = SparseState({Determinant("20"): 1.0, Determinant("02"): 0.5})

normal_result = nop.apply_to_state(state)
sparse_result = op.apply_to_state(state)

print("Normal-ordered application:")
print(normal_result.str(4))
print("\nSparseOperator application:")
print(sparse_result.str(4))
print("\nSame result:", normal_result == sparse_result)

The `@` operator is also defined as a shorthand for applying a normal-ordered operator to a `SparseState`.

In [ ]:
(nop @ state) == sparse_result